In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score, cross_val_predict

from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import CondensedNearestNeighbour, RandomUnderSampler, TomekLinks, NearMiss, EditedNearestNeighbours
from imblearn.combine import SMOTETomek
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve, auc, confusion_matrix, precision_score, recall_score, accuracy_score, f1_score, roc_auc_score

In [ ]:
# Load Dataset
raw_df = pd.read_csv('../data/full_feature_set/f5_final.csv')
raw_df

,transcript_id,transcript_position,gene_id,label,log_mean_0,log_mean_1,log_mean_2,log_std_0,log_std_1,log_std_2,...,all_dwell,log_mean_0_iqr,log_mean_1_iqr,log_mean_2_iqr,log_std_0_iqr,log_std_1_iqr,log_std_2_iqr,dwell_0_iqr,log_dwell_1_iqr,log_dwell_2_iqr
0,ENST00000000233,244,ENSG00000004059,0,-4.966140,-4.832071,-5.119335,1.316408,1.894617,1.235471,...,110.500000,-5.008637,-4.933674,-5.119335,0.463734,1.710188,1.316408,3.0,1.386294,1.098612
1,ENST00000000233,261,ENSG00000004059,0,-5.177871,-5.135349,-4.992304,1.057790,1.098612,0.978326,...,104.033333,-5.613028,-5.521461,-5.288862,0.262364,0.173953,0.279524,4.0,1.386294,1.073294
2,ENST00000000233,316,ENSG00000004059,0,-5.065620,-5.065620,-5.065620,0.974560,1.329724,0.647103,...,98.233333,-5.253344,-5.236282,-5.414851,0.190620,0.029559,-0.105361,1.0,1.131402,0.530628
3,ENST00000000233,332,ENSG00000004059,0,-4.708311,-4.917145,-5.302325,1.745716,0.968883,0.758467,...,105.800000,-4.972617,-5.178758,-5.612343,1.208960,-0.030459,-0.251672,3.0,0.832909,1.446919
4,ENST00000000233,368,ENSG00000004059,0,-4.714985,-4.556380,-4.745007,1.874874,1.733424,1.425515,...,108.466667,-4.812194,-5.004156,-5.105633,0.515813,0.763140,0.462160,3.0,0.693147,1.186318
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121833,ENST00000641834,1348,ENSG00000167747,1,-4.807286,-5.238163,-5.370888,1.163151,1.521699,1.291984,...,105.366667,-5.158555,-5.119335,-5.511511,0.239017,0.841567,0.438255,2.0,1.609438,1.410987
121834,ENST00000641834,1429,ENSG00000167747,0,-5.086437,-4.645992,-5.334981,1.305626,2.212660,1.235471,...,102.866667,-5.124364,-4.778334,-5.722965,0.698135,1.235471,0.182322,3.0,1.386294,1.098612
121835,ENST00000641834,1531,ENSG00000167747,1,-4.966140,-5.162913,-5.151623,1.342865,1.490654,0.654926,...,104.166667,-5.394828,-5.388789,-5.800844,0.563892,0.805359,-0.307885,2.0,0.810930,0.680568
121836,ENST00000641834,1537,ENSG00000167747,0,-5.020686,-4.989363,-5.065620,1.150572,1.756132,0.845868,...,105.333333,-5.276556,-5.567505,-5.779584,-0.051293,1.205971,0.122218,4.0,1.386294,1.193922


## 1. Train Test Split

In [20]:
# Step 1: create the combined column
raw_df['combined'] = raw_df['gene_id'].astype(str) + '_' + raw_df['label'].astype(str)

# Step 2: count occurrences of the combined column
counts = raw_df['combined'].value_counts()

# Step 3: where count == 1, replace with just column2
raw_df['final_group'] = raw_df.apply(
    lambda row: row['label'] if counts[row['combined']] == 1 else row['combined'],
    axis=1
)
raw_df['final_group'] = raw_df['final_group'].astype(str)

# Split into X and y
X = raw_df.drop(columns=['gene_id', 'combined'])
y = raw_df['label']

# Split into 90% vs 10%
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(X, y, stratify=raw_df['final_group'], test_size=0.1, random_state=42)
X_full_test.drop(columns=['transcript_position', 'transcript_id', 'label', 'final_group'], inplace=True)

# Split the 90 into 80% and 20%
X_subset = X_full_train.drop(columns=['final_group', 'label'])
y_subset = y_full_train

X_train_id, X_test_id, y_train, y_test = train_test_split(X_subset, y_subset, stratify=X_full_train['final_group'], test_size=0.2, random_state=42)
print(X_train_id.shape)
print(X_train_id.columns)

# To drop unique identifiers
X_train = X_train_id.drop(columns=['transcript_id', 'transcript_position'])
X_test = X_test_id.drop(columns=['transcript_id', 'transcript_position'])

display(X_train)
display(X_test)
display(y_train)
display(y_test)

(87723, 89)
Index(['transcript_id', 'transcript_position', 'log_mean_0', 'log_mean_1',
       'log_mean_2', 'log_std_0', 'log_std_1', 'log_std_2', 'dwell_0',
       'dwell_1', 'dwell_2', 'seq_5mer_0_AAAAC', 'seq_5mer_0_AAGAC',
       'seq_5mer_0_AGAAC', 'seq_5mer_0_AGGAC', 'seq_5mer_0_ATAAC',
       'seq_5mer_0_ATGAC', 'seq_5mer_0_CAAAC', 'seq_5mer_0_CAGAC',
       'seq_5mer_0_CGAAC', 'seq_5mer_0_CGGAC', 'seq_5mer_0_CTAAC',
       'seq_5mer_0_CTGAC', 'seq_5mer_0_GAAAC', 'seq_5mer_0_GAGAC',
       'seq_5mer_0_GGAAC', 'seq_5mer_0_GGGAC', 'seq_5mer_0_GTAAC',
       'seq_5mer_0_GTGAC', 'seq_5mer_0_TAAAC', 'seq_5mer_0_TAGAC',
       'seq_5mer_0_TGAAC', 'seq_5mer_0_TGGAC', 'seq_5mer_0_TTAAC',
       'seq_5mer_0_TTGAC', 'seq_5mer_1_AAACA', 'seq_5mer_1_AAACC',
       'seq_5mer_1_AAACT', 'seq_5mer_1_AGACA', 'seq_5mer_1_AGACC',
       'seq_5mer_1_AGACT', 'seq_5mer_1_GAACA', 'seq_5mer_1_GAACC',
       'seq_5mer_1_GAACT', 'seq_5mer_1_GGACA', 'seq_5mer_1_GGACC',
       'seq_5mer_1_GGACT', 'seq_5mer

,log_mean_0,log_mean_1,log_mean_2,log_std_0,log_std_1,log_std_2,dwell_0,dwell_1,dwell_2,seq_5mer_0_AAAAC,...,all_dwell,log_mean_0_iqr,log_mean_1_iqr,log_mean_2_iqr,log_std_0_iqr,log_std_1_iqr,log_std_2_iqr,dwell_0_iqr,log_dwell_1_iqr,log_dwell_2_iqr
34789,-5.177871,-4.966140,-5.188567,1.477049,1.906575,1.202972,120.00,128.00,81.20,0.0,...,109.733333,-4.887533,-5.013138,-5.724496,0.500775,1.372449,1.050822,4.000,1.386294,0.667829
87604,-4.896860,-5.302325,-5.302325,1.205971,1.860975,1.474763,103.00,123.00,82.50,0.0,...,102.833333,-5.006394,-5.369813,-5.540389,-0.192372,1.169381,1.080109,4.000,1.386294,1.280934
50294,-4.874358,-5.177871,-5.302325,0.708036,0.970779,0.900161,102.00,96.60,85.70,0.0,...,94.766667,-5.056156,-5.321586,-5.450304,-0.703198,-0.110932,-0.210721,1.000,1.178655,1.308333
26059,-5.180534,-5.077577,-5.050676,0.672944,0.777029,0.601580,92.55,93.45,84.45,0.0,...,90.150000,-5.547420,-5.857808,-5.496768,-0.245261,0.093035,-0.668455,2.475,0.587787,0.896088
100159,-5.238163,-5.022202,-5.119335,1.144223,1.735189,1.172482,108.00,121.00,86.90,0.0,...,105.300000,-5.439881,-5.494332,-5.352774,0.014889,1.027832,0.421994,5.000,1.609438,1.609438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57591,-4.990098,-4.977684,-5.031348,1.105257,1.689173,1.095273,121.00,125.00,79.40,0.0,...,108.466667,-5.070385,-5.125205,-5.357537,0.007472,1.234017,0.518794,3.000,1.098612,1.011601
24810,-5.014643,-4.903576,-4.862646,0.951658,1.223775,0.760806,104.00,96.70,88.40,0.0,...,96.366667,-5.370888,-5.442188,-5.284414,-0.186330,-0.385662,-0.494296,3.000,0.875469,0.336472
61440,-5.370888,-4.919881,-5.092931,1.572774,1.486140,1.247032,122.00,132.00,91.90,0.0,...,115.300000,-5.158555,-5.381699,-5.585999,1.482740,0.708036,0.665262,6.500,1.791759,1.280934
20688,-5.238163,-4.803621,-5.053021,1.574846,1.803359,1.380025,106.50,128.00,82.55,0.0,...,105.683333,-5.568160,-5.225067,-5.405346,0.336472,1.186318,1.397482,4.750,0.693147,1.170933


,log_mean_0,log_mean_1,log_mean_2,log_std_0,log_std_1,log_std_2,dwell_0,dwell_1,dwell_2,seq_5mer_0_AAAAC,...,all_dwell,log_mean_0_iqr,log_mean_1_iqr,log_mean_2_iqr,log_std_0_iqr,log_std_1_iqr,log_std_2_iqr,dwell_0_iqr,log_dwell_1_iqr,log_dwell_2_iqr
39450,-4.771815,-4.832071,-4.955437,1.199965,1.974775,0.559616,116.00,117.00,81.50,0.0,...,104.833333,-5.027146,-4.833326,-5.215356,1.036737,1.282322,-0.145026,5.00,1.658228,1.155308
14551,-5.148175,-5.177871,-5.370888,1.153732,2.181547,1.196948,106.00,116.00,77.90,0.0,...,99.966667,-5.370888,-5.613028,-5.629603,0.451076,1.311032,0.536493,4.00,1.386294,1.280934
107263,-4.930900,-4.908304,-5.065620,1.582067,1.905832,1.123305,119.00,116.50,77.50,0.0,...,104.333333,-5.548062,-4.744720,-5.150760,0.972671,1.091923,0.823078,2.25,1.871802,1.429114
68381,-5.302325,-4.913735,-5.019928,1.172482,1.626295,1.083499,121.00,129.00,92.05,0.0,...,114.016667,-6.102396,-5.042900,-5.074374,0.213093,0.783902,0.242946,5.00,1.321756,0.680568
10696,-4.585368,-4.737559,-5.238163,1.930071,1.876407,1.111858,116.00,125.00,88.10,0.0,...,109.700000,-4.657517,-4.970454,-5.613028,0.920283,0.959350,0.524729,4.00,1.098612,0.993252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111828,-4.830817,-5.238163,-5.119335,0.898127,1.289233,1.351962,108.00,105.00,91.15,0.0,...,101.383333,-4.776849,-5.132379,-5.455556,0.152721,0.161268,0.920283,2.00,1.945910,0.896088
99104,-5.207563,-4.748463,-5.140459,1.430311,1.728997,1.344169,123.50,127.00,83.40,0.0,...,111.300000,-5.172566,-5.417101,-5.665043,-0.002503,0.623261,1.138634,4.50,0.810930,0.974560
52540,-5.269730,-5.238163,-5.136199,1.210452,2.225704,0.968883,105.00,116.00,78.40,0.0,...,99.800000,-5.370888,-5.733726,-5.678115,1.103600,1.539552,0.303801,4.75,1.609438,0.777029
811,-4.708865,-5.210306,-5.302325,1.921325,1.470176,0.760806,134.00,95.95,84.90,0.0,...,104.950000,-5.174331,-5.830196,-5.515230,1.160804,0.696890,-0.192372,3.75,1.568616,0.896088


34789     0
87604     0
50294     0
26059     0
100159    0
         ..
57591     0
24810     0
61440     0
20688     0
15726     0
Name: label, Length: 87723, dtype: int64

39450     0
14551     0
107263    0
68381     0
10696     0
         ..
111828    0
99104     0
52540     0
811       0
99481     0
Name: label, Length: 21931, dtype: int64

## 2. Check Over-sampling/Under-sampling Techniques

PCA Plot

In [21]:
# sm = SMOTE(random_state=42)
# X_res, y_res = sm.fit_resample(X_train, y_train)

# # Standard Scaler for PCA
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X_train)
# X_res_scaled = scaler.fit_transform(X_res)

# # PCA
# pca = PCA(n_components=2)
# X_pca = pca.fit_transform(X_scaled)
# X_res_pca = pca.transform(X_res_scaled)

# # Plot PCA
# fig, ax = plt.subplots(1, 2, figsize=(12,5))
# sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y_train, ax=ax[0], alpha=0.7)
# ax[0].set_title("Before SMOTE")

# sns.scatterplot(x=X_res_pca[:,0], y=X_res_pca[:,1], hue=y_res, ax=ax[1], alpha=0.7)
# ax[1].set_title("After SMOTE")
# plt.show()

t-SNE Plot

In [22]:
# X_tsne = TSNE(n_components=2, random_state=42).fit_transform(X_train)

# plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_train, cmap='coolwarm', alpha=0.6)
# plt.title("t-SNE projection")
# plt.show()

UMAP Plot

In [ ]:
# reducer = umap.UMAP(random_state=42)
# scaled_umap = StandardScaler().fit_transform(X_train)
# X_umap_before = reducer.fit_transform(scaled_umap)

# plt.scatter(X_umap_before[:,0], X_umap_before[:,1], c=y_train, cmap='coolwarm', alpha=0.5)
# plt.title("UMAP")
# plt.show()

## 3. Model Training

### 3a. Baseline Model: Logistic Regression

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

pipe = Pipeline(steps = [
    # ('cnn', CondensedNearestNeighbour(random_state=42)),
    # ('smote', SMOTE(random_state=42, sampling_strategy=0.2)),
    # ('rus', RandomUnderSampler(random_state=42)),
    # ('smotetomek', SMOTETomek(random_state=42)),
    # ('tl', TomekLinks()),
    # ('nm', NearMiss(version=1)),
    # ('enn', EditedNearestNeighbours()),
    ('standardscaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, class_weight='balanced', max_iter=10000))
    # ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))

])

train_cv = cross_val_score(pipe, X_train, y_train, cv=cv, verbose=2, scoring='average_precision')
train_cv_mean = np.mean(train_cv)
pipe.fit(X_train, y_train)
print(f'10-Fold Cross Validation Score on training data: {train_cv_mean}')

[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   0.9s
[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   0.7s
[CV] END .................................................... total time=   0.9s
[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   0.7s
[CV] END .................................................... total time=   0.6s
10-Fold Cross Validation Score on training data: 0.44556482090747285


In [25]:
train_cv_probs = cross_val_predict(pipe, X_train, y_train, cv=cv)
train_cv_pred = (train_cv_probs>=0.5).astype(int)

print(confusion_matrix(y_train, train_cv_pred))
print(f'precision: {precision_score(y_train, train_cv_pred)}')
print(f'recall: {recall_score(y_train, train_cv_pred)}')
print(f'f1: {f1_score(y_train, train_cv_pred)}')
print(f'accuracy: {accuracy_score(y_train, train_cv_pred)}')
print(f'roc: {roc_auc_score(y_train, train_cv_probs)}')

[[70821 12807]
 [  698  3397]]
precision: 0.20963959516168848
recall: 0.8295482295482296
f1: 0.33469629045765803
accuracy: 0.8460494967112387
roc: 0.838202870693185


### 3b. Performance of Untuned Model

#### Validation Set

In [26]:
# Average Precision
y_pred_prob = pipe.predict_proba(X_test)[:, 1]
y_pred = pipe.predict(X_test)

ap_val_score = average_precision_score(y_test, y_pred_prob)
# print(f'Average Precision: {ap_val_score}')

print(confusion_matrix(y_test, y_pred))
print(f'precision: {precision_score(y_test, y_pred)}')
print(f'recall: {recall_score(y_test, y_pred)}')
print(f'f1: {f1_score(y_test, y_pred)}')
print(f'accuracy: {accuracy_score(y_test, y_pred)}')
print(f'roc: {roc_auc_score(y_test, y_pred_prob)}')

precision, recall, thresholds = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall, precision)
print(f"PR AUC: {pr_auc}")

[[17747  3200]
 [  196   788]]
precision: 0.19759277833500502
recall: 0.8008130081300813
f1: 0.3169750603378922
accuracy: 0.8451506999224841
roc: 0.9006942026741126
PR AUC: 0.4004071180806697


#### Test Set

In [27]:
# Average Precision
y_ho_pred_prob = pipe.predict_proba(X_full_test)[:, 1]
y_ho_pred = pipe.predict(X_full_test)

ap_test_score = average_precision_score(y_full_test, y_ho_pred_prob)
# print(f'Average Precision: {ap_test_score}')

print(confusion_matrix(y_full_test, y_ho_pred))
print(f'precision: {precision_score(y_full_test, y_ho_pred)}')
print(f'recall: {recall_score(y_full_test, y_ho_pred)}')
print(f'f1: {f1_score(y_full_test, y_ho_pred)}')
print(f'accuracy: {accuracy_score(y_full_test, y_ho_pred)}')
print(f'roc: {roc_auc_score(y_full_test, y_ho_pred_prob)}')

precision, recall, thresholds = precision_recall_curve(y_full_test, y_ho_pred_prob)
pr_auc = auc(recall, precision)
print(f"PR AUC: {pr_auc}")

[[10007  1781]
 [   77   319]]
precision: 0.1519047619047619
recall: 0.8055555555555556
f1: 0.25560897435897434
accuracy: 0.8475049244911359
roc: 0.9003007252710341
PR AUC: 0.35649311332198386


### 3c. XGB

In [28]:
xgb_pipe = Pipeline(steps=[
    ('standardscaler', StandardScaler()),
    ('model', XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        scale_pos_weight=1  
    ))
])


train_cv_xgb = cross_val_score(xgb_pipe, X_train, y_train, cv=cv, verbose=2, scoring='average_precision')
train_cv_xgb_mean = np.mean(train_cv_xgb)
xgb_pipe.fit(X_train, y_train)

print(f'10-Fold Cross Validation Score on training data: {train_cv_xgb_mean}')

[CV] END .................................................... total time=   1.0s
[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   0.9s
[CV] END .................................................... total time=   0.9s
[CV] END .................................................... total time=   1.0s
[CV] END .................................................... total time=   1.1s
[CV] END .................................................... total time=   0.9s
[CV] END .................................................... total time=   0.9s
[CV] END .................................................... total time=   1.0s
[CV] END .................................................... total time=   0.9s
10-Fold Cross Validation Score on training data: 0.4606986980958566


Performance of Untuned XGB

In [29]:
# Get Predictions
y_pred_prob = xgb_pipe.predict_proba(X_test)[:, 1]
y_pred = xgb_pipe.predict(X_test)

ap_val_score = average_precision_score(y_test, y_pred_prob)
# print(f'Average Precision: {ap_val_score}')

print(confusion_matrix(y_test, y_pred))
print(f'precision: {precision_score(y_test, y_pred)}')
print(f'recall: {recall_score(y_test, y_pred)}')
print(f'f1: {f1_score(y_test, y_pred)}')
print(f'accuracy: {accuracy_score(y_test, y_pred)}')
print(f'roc: {roc_auc_score(y_test, y_pred_prob)}')

precision, recall, thresholds = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall, precision)
print(f"PR AUC: {pr_auc}")

[[20687   260]
 [  665   319]]
precision: 0.5509499136442142
recall: 0.3241869918699187
f1: 0.40818937939859246
accuracy: 0.9578222607268251
roc: 0.9042784518884479
PR AUC: 0.426993153065157


In [30]:
# Get Predictions
y_ho_pred_prob = xgb_pipe.predict_proba(X_full_test)[:, 1]
y_ho_pred = xgb_pipe.predict(X_full_test)

ap_test_score = average_precision_score(y_full_test, y_ho_pred_prob)
# print(f'Average Precision: {ap_test_score}')

print(confusion_matrix(y_full_test, y_ho_pred))
print(f'precision: {precision_score(y_full_test, y_ho_pred)}')
print(f'recall: {recall_score(y_full_test, y_ho_pred)}')
print(f'f1: {f1_score(y_full_test, y_ho_pred)}')
print(f'accuracy: {accuracy_score(y_full_test, y_ho_pred)}')
print(f'roc: {roc_auc_score(y_full_test, y_ho_pred_prob)}')

precision, recall, thresholds = precision_recall_curve(y_full_test, y_ho_pred_prob)
pr_auc = auc(recall, precision)
print(f"PR AUC: {pr_auc}")

[[11648   140]
 [  265   131]]
precision: 0.4833948339483395
recall: 0.33080808080808083
f1: 0.39280359820089955
accuracy: 0.9667596848325674
roc: 0.909542703931065
PR AUC: 0.367636855595107


## 4. GridSearch: Optimized on Average Precision

In [31]:
# param_grid = [
#     {
#         'model__solver': ['liblinear'],
#         'model__penalty': ['l1', 'l2'],
#         'model__C': [0.01, 0.1, 1, 10]
#     },
#     # {
#     #     'clf__solver': ['saga'],
#     #     'clf__penalty': ['elasticnet'],
#     #     'clf__C': [0.01, 0.1, 1, 10],
#     #     'clf__l1_ratio': [0.5]
#     # }
# ]

# grid_search = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring='average_precision', verbose=3)
# grid_search.fit(X_train, y_train)
# best_lr = grid_search.best_estimator_
# best_params = grid_search.best_params_
# best_score = grid_search.best_score_

# print(f'Best Params: {best_params}')
# print(f'Best Score: {best_score}')

### 4a. Performance of Tuned Model

#### Validation Set

In [32]:
# # Get Predictions
# y_pred_prob = best_lr.predict_proba(X_test)[:, 1]
# y_pred = best_lr.predict(X_test)

# ap_val_score = average_precision_score(y_test, y_pred_prob)
# print(f'Average Precision: {ap_val_score}')

# precision, recall, thresholds = precision_recall_curve(y_test, y_pred_prob)
# pr_auc = auc(recall, precision)
# print(f"PR AUC: {pr_auc}")


#### Test Set

In [33]:
# # Get Predictions
# y_ho_pred_prob = best_lr.predict_proba(X_full_test)[:, 1]
# y_ho_pred = best_lr.predict(X_full_test)

# ap_test_score = average_precision_score(y_full_test, y_ho_pred_prob)
# print(f'Average Precision: {ap_test_score}')

# precision, recall, thresholds = precision_recall_curve(y_full_test, y_ho_pred_prob)
# pr_auc = auc(recall, precision)
# print(f"PR AUC: {pr_auc}")